In [1]:
!pip install -q transformers datasets accelerate scikit-learn


In [2]:
import torch
import random
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

dispozitiv = torch.device("cuda" if torch.cuda.is_available() else "cpu")
samanta = 42
random.seed(samanta)
np.random.seed(samanta)
torch.manual_seed(samanta)
if dispozitiv.type == "cuda":
    torch.cuda.manual_seed_all(samanta)


In [3]:
nume_model = "microsoft/codebert-base"
tokenizator = AutoTokenizer.from_pretrained(nume_model)
model = AutoModelForSequenceClassification.from_pretrained(nume_model, num_labels=2)
model.to(dispozitiv)


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [4]:
dataset_initial = load_dataset("DaniilOr/SemEval-2026-Task13", "A")
date_antrenare = dataset_initial["train"]
date_validare = dataset_initial["validation"]


README.md:   0%|          | 0.00/801 [00:00<?, ?B/s]

task_a/task_a_training_set_1.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

task_a/task_a_validation_set.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

task_a/task_a_test_set_sample.parquet:   0%|          | 0.00/593k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/500000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
lungime_max = 256
def tokenizeaza(exemple):
    texte = exemple["code"]
    lbl = exemple["label"]
    t = tokenizator(
        texte,
        truncation=True,
        padding="max_length",
        max_length=lungime_max
    )
    t["labels"] = lbl
    return t

train_proc = date_antrenare.map(tokenizeaza, batched=True, remove_columns=["code", "generator", "language"])
valid_proc = date_validare.map(tokenizeaza, batched=True, remove_columns=["code", "generator", "language"])
train_proc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_proc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


Map:   0%|          | 0/500000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [6]:
marime_batch = 32

loader_train = DataLoader(train_proc, batch_size=marime_batch, shuffle=True)
loader_valid = DataLoader(valid_proc, batch_size=marime_batch, shuffle=False)



In [ ]:
for p in model.roberta.parameters():
    p.requires_grad = False
straturi = list(model.roberta.encoder.layer)
total = len(straturi)
pas_deblocare = 4
def deblocheaza_straturi(n):
    n = max(0, min(total, n))
    start = total - n
    for i in range(total):
        strat = straturi[i]
        permite = (i >= start)
        for p in strat.parameters():
            p.requires_grad = permite


In [ ]:
deblocheaza_straturi(pas_deblocare)
lista_enc = []
lista_clf = []
for nume, p in model.named_parameters():
    if "classifier" in nume:
        lista_clf.append(p)
    else:
        lista_enc.append(p)
lr_enc = 2e-5
lr_clf = 1e-4
nr_epoci = 3
opt = AdamW(
    [
        {"params": [x for x in lista_enc if x.requires_grad], "lr": lr_enc},
        {"params": [x for x in lista_clf if x.requires_grad], "lr": lr_clf},
    ]
)
pasi_epoca = len(loader_train)
total_pasi = pasi_epoca * nr_epoci
warm = int(total_pasi * 0.06)
scheduler = get_linear_schedule_with_warmup(
    opt,
    num_warmup_steps=warm,
    num_training_steps=total_pasi
)


In [ ]:
def testeaza():
    model.eval()
    toate_pred = []
    toate_adev = []
    with torch.no_grad():
        for lot in loader_valid:
            intrari = lot["input_ids"].to(dispozitiv)
            masca = lot["attention_mask"].to(dispozitiv)
            etic = lot["labels"].to(dispozitiv)
            ies = model(input_ids=intrari, attention_mask=masca)
            scoruri = ies.logits
            pred_lot = torch.argmax(scoruri, dim=1)
            toate_pred += pred_lot.cpu().numpy().tolist()
            toate_adev += etic.cpu().numpy().tolist()

    f1_macro = f1_score(toate_adev, toate_pred, average="macro")
    return f1_macro


In [ ]:
import os
import torch

fisier_model = "codebert_finetuned_binar.pt"
cale_checkpoint = "/content/drive/MyDrive/licenta/checkpoint_codebert_taskA.pt"
cel_mai_bun = 0.0
start_epoca = 0
if os.path.exists(cale_checkpoint):
    checkpoint = torch.load(cale_checkpoint, map_location=dispozitiv)
    model.load_state_dict(checkpoint["model_state"])
    opt.load_state_dict(checkpoint["opt_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    cel_mai_bun = checkpoint["cel_mai_bun"]
    start_epoca = checkpoint["epoca"] + 1
    print("Am gasit checkpoint. Reiau de la epoca:", start_epoca + 1)
else:
    print("Nu am gasit checkpoint. Porneasc de la epoca 1.")


Nu am gasit checkpoint. Porneasc de la epoca 1.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs("/content/drive/MyDrive/licenta", exist_ok=True)


Mounted at /content/drive


In [ ]:

for ep in range(start_epoca, nr_epoci):
    nr_deblocate = (ep + 1) * pas_deblocare
    deblocheaza_straturi(nr_deblocate)
    model.train()
    lista_pierderi = []
    for lot in loader_train:
        id_uri = lot["input_ids"].to(dispozitiv)
        masca_at = lot["attention_mask"].to(dispozitiv)
        lab = lot["labels"].to(dispozitiv)
        ies = model(input_ids=id_uri, attention_mask=masca_at, labels=lab)
        loss_curent = ies.loss
        lista_pierderi.append(loss_curent.item())
        loss_curent.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        scheduler.step()
        opt.zero_grad()
    scor_f1 = testeaza()
    print("Epoca:", ep + 1,
          "pierdere_medie:", float(np.mean(lista_pierderi)),
          "F1_macro_validare:", scor_f1)
    if scor_f1 > cel_mai_bun:
        cel_mai_bun = scor_f1
        torch.save(model.state_dict(), fisier_model)
        print("Model imbunatatit. F1_macro_validare:", cel_mai_bun)
    checkpoint = {
        "epoca": ep,
        "model_state": model.state_dict(),
        "opt_state": opt.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "cel_mai_bun": cel_mai_bun,
    }
    torch.save(checkpoint, cale_checkpoint)
    print("Checkpoint salvat dupa epoca", ep + 1)


Epoca: 1 pierdere_medie: 0.06501832012975728 F1_macro_validare: 0.9896813795293021
Model imbunatatit. F1_macro_validare: 0.9896813795293021
Checkpoint salvat dupa epoca 1
Epoca: 2 pierdere_medie: 0.030396034302691232 F1_macro_validare: 0.9917943512624884
Model imbunatatit. F1_macro_validare: 0.9917943512624884
Checkpoint salvat dupa epoca 2
Epoca: 3 pierdere_medie: 0.024541230915440245 F1_macro_validare: 0.9927752983672286
Model imbunatatit. F1_macro_validare: 0.9927752983672286
Checkpoint salvat dupa epoca 3


In [ ]:
state = torch.load("codebert_finetuned_binar.pt", map_location=dispozitiv)
model.load_state_dict(state)
model.to(dispozitiv)
model.eval()




RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [ ]:
from datasets import Dataset
nr_exemple_mici = 500
valid_amestecat = date_validare.shuffle(seed=42)
nr_exemple_mici = min(nr_exemple_mici, len(valid_amestecat))
valid_mic = valid_amestecat.select(range(nr_exemple_mici))
print(len(valid_mic))
print(valid_mic[0])

500
{'code': 'cpp\nint n, m, a[100005];\nclass Solution {\npublic:\n    vector<int> canSeePersonsCount(vector<int>& h) {\n        n = h.size();\n        m = 0;\n        m = a[n-1]  -  h[n-1];\n        a[n-1] = 0;\n        for (int i = (n-2); i >= 0; --i) {\n            a[i] = m;\n            m = min(m, h[i]);\n        }\n        for (int i = (n-2); i >= 0; --i) {\n            a[i] += a[i+1];\n        }\n        for (int i = 0; i < n; ++i) {\n            h[i] = a[i];\n        }\n        return h;\n    }\n};', 'generator': 'Qwen/Qwen2.5-Coder-1.5B-Instruct', 'label': 1, 'language': 'C++'}


In [ ]:

from torch.utils.data import DataLoader
valid_mic_proc = valid_mic.map(
    tokenizeaza,
    batched=True,
    remove_columns=["code", "generator", "language"]
)
valid_mic_proc.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)
batch_mic = 32
loader_valid_mic = DataLoader(
    valid_mic_proc,
    batch_size=batch_mic,
    shuffle=False
)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
from sklearn.metrics import f1_score
import torch
def testeaza_loader(loader_valid_curent):
    model.eval()
    toate_pred = []
    toate_adev = []
    with torch.no_grad():
        for lot in loader_valid_curent:
            intrari = lot["input_ids"].to(dispozitiv)
            masca  = lot["attention_mask"].to(dispozitiv)
            etic   = lot["labels"].to(dispozitiv)

            ies = model(input_ids=intrari, attention_mask=masca)
            scoruri = ies.logits
            pred_lot = torch.argmax(scoruri, dim=1)

            toate_pred += pred_lot.cpu().numpy().tolist()
            toate_adev += etic.cpu().numpy().tolist()
    f1_macro = f1_score(toate_adev, toate_pred, average="macro")
    return f1_macro

In [ ]:
cale_model_finetuned = fisier_model
state_dict = torch.load(cale_model_finetuned, map_location=dispozitiv)
model.load_state_dict(state_dict)
model.to(dispozitiv)
f1_mic_codebert = testeaza_loader(loader_valid_mic)
print("F1 macro pe mini-validare (500 exemple) CodeBERT:", f1_mic_codebert)


F1 macro pe mini-validare (500 exemple) CodeBERT: 0.9899989998999901


In [ ]:
fragmente_cod = valid_mic["code"]
etichete_adev = valid_mic["label"]
len(fragmente_cod), len(etichete_adev)

(500, 500)

In [ ]:
!pip install -q transformers accelerate

from sklearn.metrics import f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "deepseek-ai/deepseek-coder-1.3b-instruct"
tokenizer_llm = AutoTokenizer.from_pretrained(model_name)
model_llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)
task_txt = """
Esti un clasificator. Primesti un fragment de cod si trebuie sa spui:
1 -> daca face parte din clasa pozitiva (ex: cod vulnerabil sau cu probleme)
0 -> daca este din clasa negativa (cod ok)

Raspunde strict cu un singur caracter: 0 sau 1.
"""
def clasifica_llm(fragment):
    prompt = task_txt + "\n\nFragment:\n```code\n" + fragment + "\n```\n\nEticheta:"
    x = tokenizer_llm(prompt, return_tensors="pt").to(model_llm.device)
    out = model_llm.generate(
        **x,
        max_new_tokens=4,
        do_sample=False,
        pad_token_id=tokenizer_llm.eos_token_id
    )
    txt = tokenizer_llm.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if "0" in txt and "1" not in txt:
        return 0
    if "1" in txt and "0" not in txt:
        return 1
    try:
        return int(txt[0])
    except:
        return 0
fragmente = valid_mic["code"]
etichete = valid_mic["label"]
predictii = []
for i, cod in enumerate(fragmente):
    p = clasifica_llm(cod)
    predictii.append(p)

    if (i + 1) % 50 == 0:
        print("Procesate", i + 1, "fragmente")
f1_llm = f1_score(etichete, predictii, average="macro")
print("F1 pe mini-validare (LLM open-source):", f1_llm)


Procesate 50 fragmente
Procesate 100 fragmente
Procesate 150 fragmente
Procesate 200 fragmente
Procesate 250 fragmente
Procesate 300 fragmente
Procesate 350 fragmente
Procesate 400 fragmente
Procesate 450 fragmente
Procesate 500 fragmente
F1 pe mini-validare (LLM open-source): 0.439484126984127


In [ ]:
!pip install -q transformers accelerate
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

tok = AutoTokenizer.from_pretrained(model_name)
mod = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

if tok.pad_token_id is None:
    tok.pad_token_id = tok.eos_token_id

task_txt = """
Esti un clasificator. Primesti un fragment de cod si trebuie sa spui:
1 -> daca face parte din clasa pozitiva (ex: cod vulnerabil sau cu probleme)
0 -> daca este din clasa negativa (cod ok)

Raspunde strict cu un singur caracter: 0 sau 1.
"""

def clasifica(cod):
    chat = [
        {"role": "system", "content": task_txt},
        {"role": "user", "content": f"{cod}\n\nEticheta:"}
    ]
    text = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    x = tok(text, return_tensors="pt").to(mod.device)
    y = mod.generate(**x, max_new_tokens=4, do_sample=False, pad_token_id=tok.pad_token_id)
    r = tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    if "0" in r and "1" not in r:
        return 0
    if "1" in r and "0" not in r:
        return 1
    try:
        return int(r[0])
    except:
        return 0

from sklearn.metrics import f1_score

coduri = valid_mic["code"]
lab = valid_mic["label"]
pred = []
for i, c in enumerate(coduri):
    pred.append(clasifica(c))
    if (i + 1) % 50 == 0:
        print(i + 1, "/", len(coduri))
print("F1:", f1_score(lab, pred, average="macro"))



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

50 / 500
100 / 500
150 / 500
200 / 500
250 / 500
300 / 500
350 / 500
400 / 500
450 / 500
500 / 500
F1: 0.39340184324280236


In [ ]:
!pip install -q transformers accelerate
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from sklearn.metrics import f1_score
model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"
tok = AutoTokenizer.from_pretrained(model_name)
mod = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
if tok.pad_token_id is None:
    tok.pad_token_id = tok.eos_token_id
task_txt = """
Esti un clasificator. Primesti un fragment de cod si trebuie sa spui:
1 -> daca face parte din clasa pozitiva (ex: cod vulnerabil sau cu probleme)
0 -> daca este din clasa negativa (cod ok)

Raspunde strict cu un singur caracter: 0 sau 1.
"""
def clasifica(cod):
    chat = [
        {"role": "system", "content": task_txt},
        {"role": "user", "content": f"{cod}\n\nEticheta:"}
    ]
    text = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    x = tok(text, return_tensors="pt").to(mod.device)
    y = mod.generate(**x, max_new_tokens=4, do_sample=False, pad_token_id=tok.pad_token_id)
    r = tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    if "0" in r and "1" not in r:
        return 0
    if "1" in r and "0" not in r:
        return 1
    try:
        return int(r[0])
    except:
        return 0
coduri = valid_mic["code"]
lab = valid_mic["label"]
pred = []
for i, c in enumerate(coduri):
    pred.append(clasifica(c))
    if (i + 1) % 50 == 0:
        print(i + 1, "/", len(coduri))
print("F1:", f1_score(lab, pred, average="macro"))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

50 / 500
100 / 500
150 / 500
200 / 500
250 / 500
300 / 500
350 / 500
400 / 500
450 / 500
500 / 500
F1: 0.34370799731182794


In [ ]:
import pandas as pd
fisier_csv = "/content/drive/MyDrive/licenta/fragmente_random_500.csv"

df = pd.DataFrame({
    "code": valid_mic["code"],
    "label": valid_mic["label"]
})
df.to_csv(fisier_csv, index=False)
print("salvat in:", fisier_csv)


salvat in: /content/drive/MyDrive/licenta/fragmente_random_500.csv


In [ ]:
cale_salveaza = "/content/drive/MyDrive/licenta/codebert_taskA/model_initial"
model.save_pretrained(cale_salveaza)
tokenizator.save_pretrained(cale_salveaza)


('/content/drive/MyDrive/licenta/codebert_taskA/model_initial/tokenizer_config.json',
 '/content/drive/MyDrive/licenta/codebert_taskA/model_initial/special_tokens_map.json',
 '/content/drive/MyDrive/licenta/codebert_taskA/model_initial/vocab.json',
 '/content/drive/MyDrive/licenta/codebert_taskA/model_initial/merges.txt',
 '/content/drive/MyDrive/licenta/codebert_taskA/model_initial/added_tokens.json',
 '/content/drive/MyDrive/licenta/codebert_taskA/model_initial/tokenizer.json')

In [ ]:
from datasets import DatasetDict
set_final = DatasetDict({
    "train": train_proc,
    "validation": valid_proc
})
set_final.save_to_disk("/content/drive/MyDrive/licenta/dataset_tokenizat_taskA")


Saving the dataset (0/2 shards):   0%|          | 0/500000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100000 [00:00<?, ? examples/s]

In [ ]:
print(date_test.column_names)
print(date_test[0])



['code', 'generator', 'label', 'language']
{'code': 'public Vector To(Vector o)\n        {\n            Vector direction = Direction(this, o);\n            X = direction.X;\n            Y = direction.Y;\n            Z = direction.Z;\n            return this;\n        }', 'generator': 'Human', 'label': 0, 'language': 'C#'}


In [ ]:
!pip install torchinfo

In [ ]:
from torchinfo import summary

summary(
    model,
    input_size=(1, lungime_max),
    dtypes=[torch.long],
    col_names=["input_size", "output_size", "num_params", "trainable"],
)

Layer (type:depth-idx)                                       Input Shape               Output Shape              Param #                   Trainable
RobertaForSequenceClassification                             [1, 256]                  [1, 2]                    --                        True
├─RobertaModel: 1-1                                          [1, 256]                  [1, 256, 768]             --                        True
│    └─RobertaEmbeddings: 2-1                                --                        [1, 256, 768]             --                        True
│    │    └─Embedding: 3-1                                   [1, 256]                  [1, 256, 768]             38,603,520                True
│    │    └─Embedding: 3-2                                   [1, 256]                  [1, 256, 768]             768                       True
│    │    └─Embedding: 3-3                                   [1, 256]                  [1, 256, 768]             394,752           

In [ ]:
lungime_max = 256